<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/03_PCMCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the pcmci family of causal discovery algorithms on the correctedv3 dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

OUTPUT_DIR = "/content/drive/MyDrive/ml/CORRECTEDv3/causal_graphs/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
!pip install tigramite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 6.7 MB/s eta 0:00:00


# Objective

- PCMCI+ — single pooled temporal causal discovery ,  Output : Directed lagged graph
- J-PCMCI+ — joint multi-city temporal causal discovery , Output : Joint graph

# why reduced set of features

- PCMCI+ runtime scales roughly as O(N² × T) for the skeleton phase and O(N³ × T) for the MCI test phase, where N is the number of variables and T is the time-series length.

- Going from N=28 to N=10 reduces the MCI phase by a factor of (28/10)³ ≈ 22×.

-  ParCorr conditions on linear combinations of the conditioning set — if we include both X and a monotone transform of X (like workload_causal and workload_capped), the conditioning sets become linearly dependent and the partial correlation tests lose meaning.

In [4]:
import pandas as pd
import numpy as np
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

# 1. Load the dataset
dataset_path = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'
df = pd.read_parquet(dataset_path)

parcorr - The theoretical backing is the faithfulness condition (Spirtes et al., 2000): PCMCI+ assumes the graph is faithful to the distribution, but including functional redundancies violates this because the conditional independences you observe are artefacts of the transforms, not of the causal structure.

- eta_mins                     ← target
- workload_causal              ← primary queue-depth cause
- batch_size                   ← dispatch volume
- batch_rank_dispatch          ← position in batch sequence  
- pickup_destination_distance  ← physical constraint
speed_mean_15m               ← pre-delivery mobility state
spatial_congestion_norm      ← area-level congestion (one version only)
WSI                          ← weather composite
hour_sin                     ← continuous time-of-day (exogenous driver)
hour_cos                     ← paired cyclical component

In [5]:
# 2. Define the features requested
# requested_features = [
#     'workload_causal', 'workload_capped', 'high_load', 'overloaded',
#     'pickup_destination_distance', 'batch_size', 'batch_rank_dispatch',
#     'batch_rank_capped', 'late_batch', 'extreme_batch', 'hour_sin', 'hour_cos',
#     'day_sin', 'day_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve',
#     'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7',
#     'spatial_congestion_norm', 'courier_local_load', 'WSI', 'precipitation',
#     'temperature_2m', 'windspeed_10m', 'is_trajectory_available', 'typecode_cb', 'eta_mins'
# ]  original set

PCMCI_FEATURES = [
    "eta_mins",                      # target
    "workload_causal",               # queue depth at dispatch
    "batch_size",                    # orders dispatched simultaneously
    "batch_rank_dispatch",           # position in dispatch sequence
    "pickup_destination_distance",   # physical constraint (invariant)
    "speed_mean_15m",                # pre-delivery mobility state
    "spatial_congestion_norm",       # area-level delivery density (z-scored)
    "WSI",                           # weather composite
    "hour_sin",                      # time-of-day: continuous cyclical
    "hour_cos",                      # paired with hour_sin
]

In [6]:
OPTIONAL = ["delivery_sequence_daily", "day_sin", "day_cos"]

MIN_OBSERVATIONS = 50   # minimum deliveries per courier to include in analysis
TAU_MAX          = 3    # 3 deliveries back; increase to 4 if ACF shows longer memory
PC_ALPHA         = 0.01 # conservative; standard for causal discovery
ALPHA_MCI        = 0.01 # MCI test threshold

In [7]:
def run_pcmci_for_city(city_name: str, df: pd.DataFrame) -> dict:
    """
    Run PCMCI+ per courier within a city, then aggregate results.

    Each courier is treated as an independent realisation of the same
    data-generating process. Running per-courier avoids cross-courier
    spurious lag links while allowing aggregation via edge frequency.

    Returns
    -------
    dict  {courier_id: pcmci_result} for couriers with sufficient observations
    """
    # Use only available features
    features = [f for f in PCMCI_FEATURES if f in df.columns]
    features += [f for f in OPTIONAL if f in df.columns]

    # Structural fill for GPS nulls before analysis
    # (consistent with add_gps_missingness_flag design)
    df = df.copy()
    for col in ["speed_mean_15m", "speed_std_15m", "distance_travelled_15m"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Remaining nulls in selected features — fill with column mean
    # (weather ASOF join occasionally misses boundary rows)
    df[features] = df[features].fillna(df[features].mean())

    couriers = df["delivery_user_id"].unique()
    results  = {}
    parcorr  = ParCorr(significance="analytic")

    for courier_id in couriers:
        courier_df = (
            df[df["delivery_user_id"] == courier_id]
            .sort_values("receipt_time")
            [features]
        )

        if len(courier_df) < MIN_OBSERVATIONS:
            continue  # insufficient time-series length for tau_max=3

        data_array  = courier_df.values.astype(float)
        tg_dataframe = pp.DataFrame(
            data_array,
            var_names=features,
            datatime=np.arange(len(courier_df)),
        )

        pcmci = PCMCI(
            dataframe=tg_dataframe,
            cond_ind_test=parcorr,
            verbosity=0,  # suppress per-courier output
        )

        result = pcmci.run_pcmciplus(
            tau_min=0,       # include contemporaneous (tau=0) links
            tau_max=TAU_MAX,
            pc_alpha=PC_ALPHA,
        )
        results[courier_id] = result

    print(f"  {city_name}: {len(results)} couriers with ≥{MIN_OBSERVATIONS} obs")
    return results, features

# -------------------------------------------

def aggregate_edge_frequency(
    city_results: dict,
    features: list,
    alpha_level: float = ALPHA_MCI,
) -> np.ndarray:
    """
    Compute edge presence frequency across couriers (bootstrap consensus).

    For each edge (i, j, tau), count what fraction of couriers show
    a significant link. This is the P(e) bootstrap stability score from
    Stage 5 of the thesis pipeline.

    Returns
    -------
    np.ndarray  shape (N, N, tau_max+1) — edge frequency matrix
    """
    N       = len(features)
    freq    = np.zeros((N, N, TAU_MAX + 1))
    n_total = len(city_results)

    if n_total == 0:
        return freq

    for result in city_results.values():
        p_matrix = result["p_matrix"]  # shape (N, N, tau_max+1)
        significant = (p_matrix <= alpha_level).astype(float)
        freq += significant

    return freq / n_total  # proportion of couriers with significant edge




In [8]:
# ── Run per city ──────────────────────────────────────────────────────────────
city_graphs   = {}
city_features = {}

# Group the existing 'df' by city instead of looking for external files
for city_name, df_city in df.groupby('city'):
    print(f"\n{'='*55}")
    print(f"  PCMCI+ — {city_name}")
    print(f"{'='*55}")

    # Run the analysis using the grouped dataframe
    results, feat_list = run_pcmci_for_city(city_name, df_city)

    freq_matrix = aggregate_edge_frequency(results, feat_list)
    city_graphs[city_name]   = freq_matrix
    city_features[city_name] = feat_list

    # Save per-city results
    city_slug = city_name.lower().replace(' ', '_')
    np.save(f"{OUTPUT_DIR}/pcmci_freq_{city_slug}.npy", freq_matrix)
    np.save(f"{OUTPUT_DIR}/pcmci_features_{city_slug}.npy", np.array(feat_list))
    print(f"  Saved frequency matrix: shape {freq_matrix.shape}")


# ── Print consensus edges (appear in >50% of couriers) ───────────────────────
CONSENSUS_THRESHOLD = 0.50

for city_name, freq in city_graphs.items():
    feat = city_features[city_name]
    print(f"\n--- Consensus edges (P(e) > {CONSENSUS_THRESHOLD}) — {city_name} ---")
    N = len(feat)
    found = False
    for i in range(N):
        for j in range(N):
            for tau in range(TAU_MAX + 1):
                if freq[i, j, tau] > CONSENSUS_THRESHOLD:
                    direction = "→" if tau > 0 else "↔"
                    lag_str   = f"(lag {tau})" if tau > 0 else "(contemp.)"
                    print(f"  {feat[i]} {direction} {feat[j]}  {lag_str}"
                          f"  P(e) = {freq[i,j,tau]:.2f}")
                    found = True
    if not found:
        print("  No consensus edges at this threshold.")


  PCMCI+ — Chongqing
  Chongqing: 119 couriers with ≥50 obs
  Saved frequency matrix: shape (12, 12, 4)

  PCMCI+ — Hangzhou
  Hangzhou: 141 couriers with ≥50 obs
  Saved frequency matrix: shape (12, 12, 4)

  PCMCI+ — Shanghai


/usr/local/lib/python3.12/dist-packages/tigramite/independence_tests/parcorr.py:146: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val, _ = stats.pearsonr(x_vals, y_vals)
/usr/local/lib/python3.12/dist-packages/tigramite/independence_tests/parcorr.py:146: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val, _ = stats.pearsonr(x_vals, y_vals)
/usr/local/lib/python3.12/dist-packages/tigramite/independence_tests/parcorr.py:146: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val, _ = stats.pearsonr(x_vals, y_vals)


  Shanghai: 95 couriers with ≥50 obs
  Saved frequency matrix: shape (12, 12, 4)

--- Consensus edges (P(e) > 0.5) — Chongqing ---
  workload_causal → workload_causal  (lag 1)  P(e) = 0.98
  workload_causal ↔ batch_size  (contemp.)  P(e) = 0.64
  workload_causal ↔ delivery_sequence_daily  (contemp.)  P(e) = 0.92
  batch_size ↔ workload_causal  (contemp.)  P(e) = 0.64
  batch_size → batch_size  (lag 1)  P(e) = 0.96
  batch_size ↔ batch_rank_dispatch  (contemp.)  P(e) = 0.98
  batch_rank_dispatch ↔ batch_size  (contemp.)  P(e) = 0.98
  spatial_congestion_norm → spatial_congestion_norm  (lag 1)  P(e) = 0.93
  spatial_congestion_norm ↔ hour_sin  (contemp.)  P(e) = 0.54
  WSI → WSI  (lag 1)  P(e) = 1.00
  hour_sin ↔ spatial_congestion_norm  (contemp.)  P(e) = 0.54
  hour_sin → hour_sin  (lag 1)  P(e) = 0.70
  hour_sin ↔ hour_cos  (contemp.)  P(e) = 0.91
  hour_sin ↔ delivery_sequence_daily  (contemp.)  P(e) = 0.82
  hour_cos ↔ hour_sin  (contemp.)  P(e) = 0.91
  hour_cos → hour_cos  (lag 1)

In [9]:
!pip install tigramite
import matplotlib.pyplot as plt
from tigramite import plotting as tp
import numpy as np

# Visualize results for cities that processed successfully
for city_name, freq in city_graphs.items():
    feat = city_features[city_name]
    print(f"\nGraph for {city_name}:")

    # Plotting contemporaneous and lagged links using the frequency matrix
    tp.plot_graph(
        val_matrix=freq,
        var_names=feat,
        show_colorbar=True,
        node_size=0.5,
        arrow_linewidth=2.0,
        label_fontsize=10
    )
    plt.show()


Graph for Chongqing:


TypeError: plot_graph() missing 1 required positional argument: 'graph'